<a href="https://colab.research.google.com/github/saraxfl/Notebook/blob/main/Collab6_Bloque2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns, sklearn, torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

torch.set_num_threads(1)  # same results run after run on CPU
torch.use_deterministic_algorithms(True)
plt.rcParams['figure.figsize'] = (10, 4)

print('torch', torch.__version__, '| sklearn', sklearn.__version__)


torch 2.11.0+cpu | sklearn 1.6.1


In [ ]:

flights = sns.load_dataset('flights')  # 144 rows: year, month, passengers (Jan 1949 - Dec 1960)

y = pd.Series(
    flights['passengers'].values,
    index=pd.date_range('1949-01-01', periods=144, freq='MS'),
    name='passengers'
)

assert len(y) == 144 and y.sum() == 40363

train, test = y[:'1958'], y['1959':]

H = len(test)  # 24 months ahead, all forecast in December 1958

def rmse(f, a):
    return float(np.sqrt(np.mean((np.asarray(f, float) - np.asarray(a, float)) ** 2)))

def mape(f, a):
    return float(100 * np.mean(np.abs((np.asarray(f, float) - np.asarray(a, float)) / np.asarray(a, float))))

results = {}

snaive = np.tile(train.values[-12:], 2)  # 1958 copied into 1959 and 1960

results['Seasonal naive'] = (
    rmse(snaive, test),
    mape(snaive, test)
)


In [ ]:


def make_windows(s, p):
    s = np.asarray(s, float)
    X = np.array([s[i - p:i] for i in range(p, len(s))])
    return X, s[p:]


In [ ]:

def recursive_forecast(model, history, p, h):
    hist = list(np.asarray(history, float))
    out = []
    for _ in range(h):
        nxt = float(model.predict(np.array(hist[-p:]).reshape(1, -1))[0])
        out.append(nxt)
        hist.append(nxt)
    return np.array(out)


def z_of(Lser):
    Lser = np.asarray(Lser, float)
    return Lser[13:] - Lser[12:-1] - Lser[1:-12] + Lser[:-13]


def undo(z_future, L_hist):
    Lh = list(L_hist)
    for zt in z_future:
        Lh.append(zt + Lh[-1] + Lh[-12] - Lh[-13])
    return np.array(Lh[len(L_hist):])


In [ ]:

p = 13
L = np.log(y.values)
Ltr = L[:len(train)]

ztr = z_of(Ltr)

mz, sz = ztr.mean(), ztr.std()

Xz, tz = make_windows((ztr - mz) / sz, p)

f_zmean = np.exp(undo(np.full(H, mz), Ltr))

results['Transform, no learning'] = (
    rmse(f_zmean, test),
    mape(f_zmean, test)
)


lin = LinearRegression().fit(Xz, tz)
f_lin = np.exp(undo(
    recursive_forecast(lin, (ztr - mz) / sz, p, H) * sz + mz,
    Ltr
))
results['Linear AR(13)'] = (rmse(f_lin, test), mape(f_lin, test))

for k, (r, m) in results.items():
    print(f'{k:31s} RMSE {r:6.1f}   MAPE {m:5.1f}%')

assert abs(results['Seasonal naive'][0] - 77.0) < 0.05 and abs(results['Linear AR(13)'][0] - 67.6) < 0.05



Seasonal naive                  RMSE   77.0   MAPE  15.5%
Transform, no learning          RMSE   76.8   MAPE  15.4%
Linear AR(13)                   RMSE   67.6   MAPE  13.2%


In [ ]:
def to_seq(X):                                    # (n, 13) table  ->  (n, 13, 1) tensor of sequences
    return torch.tensor(np.asarray(X, float), dtype=torch.float32).unsqueeze(-1)

def to_target(t):
    return torch.tensor(np.asarray(t, float), dtype=torch.float32)

print('table:', Xz.shape, '->  sequences:', tuple(to_seq(Xz).shape))

table: (94, 13) ->  sequences: (94, 13, 1)


## Entrenamiento de LSTM y GRU

Las redes reciben 13 meses de la serie transformada y predicen el siguiente. Elijo el número de épocas con validaciones temporales en 1956, 1957 y 1958. En cada validación uso los meses anteriores para estimar la media y la desviación y entrenar. Reservo todo 1959–1960 para el pronóstico recursivo final de 24 meses.

[Ejercicio de referencia](https://github.com/Kei300/P2-IA-Python-Libraries-Statistics/blob/main/6_LSTM_AirPassengers.ipynb)

In [ ]:
class PredictorMensual(nn.Module):
    def __init__(self, arquitectura, unidades=16):
        super().__init__()
        capa = nn.LSTM if arquitectura == "LSTM" else nn.GRU
        self.recurrente = capa(1, unidades, batch_first=True)
        self.salida = nn.Linear(unidades, 1)
        if arquitectura == "LSTM":
            # Favorezco que la compuerta de olvido conserve información al inicio.
            with torch.no_grad():
                self.recurrente.bias_hh_l0[unidades:2 * unidades].add_(1.0)

    def forward(self, ventanas):
        secuencia, _ = self.recurrente(ventanas)
        return self.salida(secuencia[:, -1]).flatten()


def preparar_validaciones():
    fechas = train.index[13:]
    conjuntos = []
    for anio in (1956, 1957, 1958):
        inicio = int(fechas.searchsorted(pd.Timestamp(anio, 1, 1)))
        fin = int(fechas.searchsorted(pd.Timestamp(anio + 1, 1, 1)))
        previos = ztr[:inicio]
        normalizada = (ztr[:fin] - previos.mean()) / previos.std()
        ventanas, objetivos = make_windows(normalizada, p)
        corte = inicio - p
        conjuntos.append((
            to_seq(ventanas[:corte]), to_target(objetivos[:corte]),
            to_seq(ventanas[corte:]), to_target(objetivos[corte:]),
        ))
    return conjuntos


validaciones = preparar_validaciones()
X_total, y_total = to_seq(Xz), to_target(tz)
epocas_maximas = 500

In [ ]:
def entrenar(tipo, semilla, entradas, objetivos, epocas, validacion=None):
    torch.manual_seed(semilla)
    red = PredictorMensual(tipo)
    optim = torch.optim.Adam(red.parameters(), lr=.001, weight_decay=.0001)
    errores = []
    for _ in range(epocas):
        red.train()
        optim.zero_grad()
        perdida = (red(entradas) - objetivos).square().mean()
        perdida.backward()
        nn.utils.clip_grad_norm_(red.parameters(), max_norm=1.)
        optim.step()
        if validacion is not None:
            red.eval()
            with torch.inference_mode():
                x_val, y_val = validacion
                errores.append((red(x_val) - y_val).square().mean().item())
    return red, np.asarray(errores)


def evaluar_red(tipo, semilla):
    curvas = []
    for x_sub, y_sub, x_val, y_val in validaciones:
        _, historial = entrenar(tipo, semilla, x_sub, y_sub, epocas_maximas, (x_val, y_val))
        curvas.append(historial)
    error_validacion = np.mean(curvas, axis=0)
    epoca = int(error_validacion.argmin()) + 1
    red, _ = entrenar(tipo, semilla, X_total, y_total, epoca)

    historial = list((ztr - mz) / sz)
    red.eval()
    with torch.inference_mode():
        for _ in range(H):
            ventana = torch.tensor(historial[-p:], dtype=torch.float32).reshape(1, p, 1)
            historial.append(red(ventana).item())
    cambios = np.asarray(historial[-H:]) * sz + mz
    return np.exp(undo(cambios, Ltr)), epoca, error_validacion


lstm_primera = evaluar_red("LSTM", 0)
f_lstm, epoca, errores = lstm_primera
results["LSTM (semilla 0)"] = (rmse(f_lstm, test), mape(f_lstm, test))
print(f"LSTM semilla 0 · {epoca} épocas · RMSE {rmse(f_lstm, test):.1f} · MAPE {mape(f_lstm, test):.1f}%")
plt.plot(np.arange(1, len(errores) + 1), errores)
plt.axvline(epoca, linestyle="--", color="tab:red")
plt.xlabel("Época")
plt.ylabel("MSE medio de validación")
plt.title("Selección de épocas para LSTM, semilla 0")
plt.show()

In [ ]:
def pronostico_mlp(semilla):
    modelo = MLPRegressor(hidden_layer_sizes=(32,), alpha=.001, max_iter=5000, random_state=semilla)
    modelo.fit(Xz, tz)
    escalados = recursive_forecast(modelo, (ztr - mz) / sz, p, H)
    return np.exp(undo(escalados * sz + mz, Ltr))


def media_geometrica(pronosticos):
    return np.exp(np.log(pronosticos).mean(axis=0))


semillas = range(10)
corridas = {}
promedios = {}
resumen_semillas = []
for tipo in ("LSTM", "GRU"):
    ensayos = [
        lstm_primera if tipo == "LSTM" and semilla == 0 else evaluar_red(tipo, semilla)
        for semilla in semillas
    ]
    corridas[tipo] = np.stack([pronostico for pronostico, _, _ in ensayos])
    promedios[tipo] = media_geometrica(corridas[tipo])
    for semilla, (pronostico, epocas, _) in zip(semillas, ensayos):
        resumen_semillas.append((tipo, semilla, epocas, rmse(pronostico, test)))

corridas["MLP"] = np.stack([pronostico_mlp(semilla) for semilla in semillas])
promedios["MLP"] = media_geometrica(corridas["MLP"])
for tipo, pronostico in promedios.items():
    results[f"{tipo}, ensamble de 10 semillas"] = (rmse(pronostico, test), mape(pronostico, test))

por_semilla = pd.DataFrame(resumen_semillas, columns=["Modelo", "Semilla", "Épocas", "RMSE"])
display(por_semilla.groupby("Modelo")["RMSE"].agg(["min", "median", "max"]).round(1))
tabla = pd.DataFrame.from_dict(results, orient="index", columns=["RMSE", "MAPE (%)"]).round(1)
display(tabla)

In [ ]:
fig, eje = plt.subplots(figsize=(11, 4))
eje.plot(test.index, corridas["LSTM"].T, color="tab:blue", alpha=.12)
for (tipo, pronostico), color in zip(promedios.items(), ["tab:blue", "tab:green", "tab:purple"]):
    eje.plot(test.index, pronostico, color=color, label=f"{tipo}, ensamble")
eje.plot(test.index, f_lin, linestyle="--", color="tab:orange", label="AR(13)")
eje.plot(test.index, test, color="black", linewidth=2, label="Observado")
eje.set(title="AirPassengers · pronóstico a 24 meses desde diciembre de 1958",
        ylabel="Pasajeros")
eje.legend()
plt.show()